In [1]:
import sys

sys.path.append('/home/ABTLUS/juventino.fonseca/repos/cax-scripts')

In [2]:
import time

import epics
from caxscripts.h5file import HDF5File

import numpy as np
import matplotlib.pyplot as plt

Script for scan [type of scan ...]

# Parameters

Devices PVs

In [3]:
PV_DVF1_NY = "CAX:A:BASLER01:image1:ArraySize0_RBV"
PV_DVF1_NX = "CAX:A:BASLER01:image1:ArraySize1_RBV"
PV_DVF1_IMG = "CAX:A:BASLER01:image1:ArrayData"
PV_DVF1_EXPOSURE_TIME_SP = "CAX:A:BASLER01:cam1:AcquireTime"
PV_DVF1_ACQUIRE_PERIOD_SP = "CAX:A:BASLER01:cam1:AcquirePeriod"
PV_DVF1_EXPOSURE_TIME_RBV = "CAX:A:BASLER01:cam1:AcquireTime_RBV"
PV_DVF1_ACQUIRE_PERIOD_RBV = "CAX:A:BASLER01:cam1:AcquirePeriod_RBV"

PV_DVF2_NY = "CAX:B:BASLER01:image1:ArraySize0_RBV"
PV_DVF2_NX = "CAX:B:BASLER01:image1:ArraySize1_RBV"
PV_DVF2_IMG = "CAX:B:BASLER01:image1:ArrayData"
PV_DVF2_EXPOSURE_TIME_SP = "CAX:B:BASLER01:cam1:AcquireTime"
PV_DVF2_ACQUIRE_PERIOD_SP = "CAX:B:BASLER01:cam1:AcquirePeriod"
PV_DVF2_EXPOSURE_TIME_RBV = "CAX:B:BASLER01:cam1:AcquireTime_RBV"
PV_DVF2_ACQUIRE_PERIOD_RBV = "CAX:B:BASLER01:cam1:AcquirePeriod_RBV"

PV_SLIT1_TOP_SP = "CAX:A:PP02:A.VAL"
PV_SLIT1_BOTTOM_SP = "CAX:A:PP02:B.VAL"
PV_SLIT1_LEFT_SP = "CAX:A:PP02:C.VAL"
PV_SLIT1_RIGHT_SP = "CAX:A:PP02:D.VAL"
PV_SLIT1_TOP_RBV = "CAX:A:PP02:A.RBV"
PV_SLIT1_BOTTOM_RBV = "CAX:A:PP02:B.RBV"
PV_SLIT1_LEFT_RBV = "CAX:A:PP02:C.RBV"
PV_SLIT1_RIGHT_RBV = "CAX:A:PP02:D.RBV"

Execution variables

In [8]:
# robust slit movement
THRESHOLD = 0.01
TRIALS = 3
DELAY = 4
COUNT_LIM = 5

# square sizes
L = 0.5
# image dimensions
NX1 = epics.caget(PV_DVF1_NX)
NY1 = epics.caget(PV_DVF1_NY)
NX2 = epics.caget(PV_DVF2_NX)
NY2 = epics.caget(PV_DVF2_NY)

# Functions

Useful functions for the scan

In [5]:
def move_slit(pv_sp, setpoint, pv_rbv, threshold=THRESHOLD, max_count=COUNT_LIM, delay=DELAY):

    try:
        epics.caput(pv_sp, setpoint)
    except Exception as err:
        raise IOError(f' PUT error: pv, pos = ({epics.caget(pv_rbv)} \n {err}') 
    
    # Check for acknowledgement. Avoid endless loop if command is not properly received.
    icount = 0
    current_value = epics.caget(pv_rbv)
    while abs(current_value - setpoint) > threshold:
        time.sleep(delay)
        current_value = epics.caget(pv_rbv)
        print('Dif: {:} \t New pos: {:} \t Curr: {:}'.format(abs(current_value - setpoint), setpoint, current_value))
        print('Slit is Moving {:.2f}'.format(current_value), end='\r')

        if icount >= max_count:
            print(f'\n WARNING: maximum number of trials to move {pv_sp} reached.'
                    f'\n Current position: {current_value}')
            return False

        icount += 1

    return True

def move_robust_slit(pv_sp, setpoint, pv_rbv, threshold=THRESHOLD, max_count=COUNT_LIM, delay=DELAY, trials=TRIALS):
    ctrials = 0
    status = False
    
    try:
        while ctrials < trials and not status:
            status = move_slit(pv_sp, setpoint, pv_rbv, threshold=THRESHOLD, max_count=COUNT_LIM, delay=DELAY)

            ctrials += 1
        current_value = epics.caget(pv_rbv)
        if not status:
            raise Exception('WARNING: maximum number of trials to move{pv_sp} reached.'
                    f'\n Current position: {current_value}')
        return True
    except Exception:
        print('Not moved')
        return False
    

def open_all_slits():
    epics.caput(PV_SLIT1_TOP_SP,19.7-0.04)
    epics.caput(PV_SLIT1_BOTTOM_SP,35.8)
    epics.caput(PV_SLIT1_LEFT_SP,43.6-0.04)
    epics.caput(PV_SLIT1_RIGHT_SP,47.2)

# Scan

## parameters

In [6]:
pos_top_open = 19.7 - 0.04
pos_bottom_open = 35.8
pos_left_open = 43.6 - 0.04
pos_right_open = 47.2

rangex = 1.3 + 0.1
rangey = 4.8

proport = rangey/rangex

## study

In [ ]:
# all opened #
epics.caput(PV_SLIT1_TOP_SP,19.7)
epics.caput(PV_SLIT1_BOTTOM_SP,35.8)
epics.caput(PV_SLIT1_LEFT_SP,43.6)
epics.caput(PV_SLIT1_RIGHT_SP,47.2)

In [ ]:
# ranges #
# epics.caput(PV_SLIT1_TOP_SP,19.7-4.8)
# epics.caput(PV_SLIT1_BOTTOM_SP,35.8-4.9)
# epics.caput(PV_SLIT1_LEFT_SP,43.6-1.3)
# epics.caput(PV_SLIT1_RIGHT_SP,47.2-1.3)

In [ ]:
# slits 1 limits for the light beam
# top    : 19.7 com 4.8 de range, ou seja, de 14.9 a 19.7
# bottom : 35.8 com 4.8 de range, ou seja, de 31.0 a 35.8
# left   : 43.6 com 1.3 de range, ou seja, de 42.3 a 43.6
# right  : 47.2 com 1.3 de range, ou seja, de 45.9 a 47.2

In [ ]:
# proportion between ranges #
proport

3.692307692307692

In [ ]:
# line movement: y
epics.caput(PV_SLIT1_TOP_SP,pos_top_open-rangey+2)
epics.caput(PV_SLIT1_BOTTOM_SP,pos_bottom_open-2)

1

In [195]:
# line movement: x
epics.caput(PV_SLIT1_LEFT_SP,pos_left_open-0.6)
epics.caput(PV_SLIT1_RIGHT_SP,pos_right_open-rangex+0.6)

1

In [ ]:
# square movement #

epics.caput(PV_SLIT1_TOP_SP,pos_top_open-rangey+L+1.7)
epics.caput(PV_SLIT1_BOTTOM_SP,pos_bottom_open-1.7)

epics.caput(PV_SLIT1_LEFT_SP,pos_left_open-0.5)
epics.caput(PV_SLIT1_RIGHT_SP,pos_right_open-rangex+L+0.5)

1

In [ ]:
# max dx: -rangex+L+dx=0 => dx=rangex-L
# max dy: -rangey+L+dy=0 => dy=rangey-L

In [ ]:
# start square #

epics.caput(PV_SLIT1_TOP_SP,pos_top_open-rangey+L)
epics.caput(PV_SLIT1_BOTTOM_SP,pos_bottom_open)

epics.caput(PV_SLIT1_LEFT_SP,pos_left_open)
epics.caput(PV_SLIT1_RIGHT_SP,pos_right_open-rangex+L)

1

## execution of scan

In [ ]:
# file = HDF5File(filename='square_scan_1806_1.h5') #,filedir=)

# file.save_metadata(metadata_dict={
#     'exposure_time': epics.caget(PV_DVF2_EXPOSURE_TIME_RBV),
#     'slit_top': epics.caget(PV_SLIT1_TOP_RBV),
#     'slit_bottom': epics.caget(PV_SLIT1_BOTTOM_RBV),
#     'slit_left': epics.caget(PV_SLIT1_LEFT_RBV),
#     'slit_right': epics.caget(PV_SLIT1_RIGHT_RBV),
# })

In [10]:
file = HDF5File(filename='square_scan_1806_2.h5') #,filedir=)

file.save_metadata(metadata_dict={
    'exposure_time': epics.caget(PV_DVF2_EXPOSURE_TIME_RBV),
    'slit_top': epics.caget(PV_SLIT1_TOP_RBV),
    'slit_bottom': epics.caget(PV_SLIT1_BOTTOM_RBV),
    'slit_left': epics.caget(PV_SLIT1_LEFT_RBV),
    'slit_right': epics.caget(PV_SLIT1_RIGHT_RBV),
})

In [15]:
Ndxs = 7
dxs = np.linspace(0,rangex-L,Ndxs)
dys = np.linspace(0,rangey-L,int(proport*Ndxs))

In [ ]:
# t0 = time.time()

# for i, dy in enumerate(dys):

#     move_robust_slit(pv_sp=PV_SLIT1_TOP_SP, pv_rbv=PV_SLIT1_TOP_RBV,       setpoint=pos_top_open-rangey+L+dy)
#     move_robust_slit(pv_sp=PV_SLIT1_BOTTOM_SP, pv_rbv=PV_SLIT1_BOTTOM_RBV, setpoint=pos_bottom_open-dy)
#     move_robust_slit(pv_sp=PV_SLIT1_LEFT_SP, pv_rbv=PV_SLIT1_LEFT_RBV,     setpoint=pos_left_open)
#     move_robust_slit(pv_sp=PV_SLIT1_RIGHT_SP, pv_rbv=PV_SLIT1_RIGHT_RBV,   setpoint=pos_right_open-rangex+l)

#     for j, dx in enumerate(dxs):

#         print(f'scan step x {j+1}/{len(dxs)} and step y {i+1}/{len(dys)}')

#         move_robust_slit(pv_sp=PV_SLIT1_LEFT_SP, pv_rbv=PV_SLIT1_LEFT_RBV,     setpoint=pos_left_open-dx)
#         move_robust_slit(pv_sp=PV_SLIT1_RIGHT_SP, pv_rbv=PV_SLIT1_RIGHT_RBV,   setpoint=pos_right_open-rangex+l+dx)

#         img1 = np.array(epics.caget(PV_DVF1_IMG))
#         img1 = img1.reshape(NX1,NY1)
        
#         img2 = np.array(epics.caget(PV_DVF2_IMG))
#         img2 = img2.reshape(NX2,NY2)

#         metadata = {
#             'slit_top'    : epics.caget(PV_SLIT1_TOP_RBV),
#             'slit_bottom' : epics.caget(PV_SLIT1_BOTTOM_RBV),
#             'slit_left'   : epics.caget(PV_SLIT1_LEFT_RBV),
#             'slit_right'  : epics.caget(PV_SLIT1_RIGHT_RBV)
#         }

#         file.save_dataset(dsetname=f'dvf1-{i}-{j}',dsetmetadata=metadata,dsetdata=img1)
#         file.save_dataset(dsetname=f'dvf2-{i}-{j}',dsetmetadata=metadata,dsetdata=img2)

# open_all_slits()

# dt = time.time()-t0
# print(f'ellapsed time [s]: {dt}')

scan step x 1/10 and step y 1/36
scan step x 2/10 and step y 1/36
Dif: 3.472222221745369e-05 	 New pos: 43.43777777777778 	 Curr: 43.4378125
Dif: 3.472222221745369e-05 	 New pos: 46.22222222222222 	 Curr: 46.222187500000004
scan step x 3/10 and step y 1/36
Dif: 8.680555552587066e-06 	 New pos: 43.315555555555555 	 Curr: 43.315546875
Dif: 8.680555552587066e-06 	 New pos: 46.34444444444445 	 Curr: 46.344453125
scan step x 4/10 and step y 1/36
Dif: 2.6041666664866625e-05 	 New pos: 43.193333333333335 	 Curr: 43.193359375
Dif: 2.6041666671972052e-05 	 New pos: 46.46666666666667 	 Curr: 46.466640625
scan step x 5/10 and step y 1/36
Dif: 1.736111111227956e-05 	 New pos: 43.071111111111115 	 Curr: 43.07109375
Dif: 1.736111111227956e-05 	 New pos: 46.58888888888889 	 Curr: 46.58890625
scan step x 6/10 and step y 1/36
Dif: 1.736111111227956e-05 	 New pos: 42.94888888888889 	 Curr: 42.94890625
Dif: 1.736111111227956e-05 	 New pos: 46.711111111111116 	 Curr: 46.71109375
scan step x 7/10 and step 

In [16]:
t0 = time.time()

for i, dy in enumerate(dys):

    move_robust_slit(pv_sp=PV_SLIT1_TOP_SP, pv_rbv=PV_SLIT1_TOP_RBV,       setpoint=pos_top_open-rangey+L+dy)
    move_robust_slit(pv_sp=PV_SLIT1_BOTTOM_SP, pv_rbv=PV_SLIT1_BOTTOM_RBV, setpoint=pos_bottom_open-dy)
    move_robust_slit(pv_sp=PV_SLIT1_LEFT_SP, pv_rbv=PV_SLIT1_LEFT_RBV,     setpoint=pos_left_open)
    move_robust_slit(pv_sp=PV_SLIT1_RIGHT_SP, pv_rbv=PV_SLIT1_RIGHT_RBV,   setpoint=pos_right_open-rangex+L)

    for j, dx in enumerate(dxs):

        print(f'scan step x {j+1}/{len(dxs)} and step y {i+1}/{len(dys)}')

        move_robust_slit(pv_sp=PV_SLIT1_LEFT_SP, pv_rbv=PV_SLIT1_LEFT_RBV,     setpoint=pos_left_open-dx)
        move_robust_slit(pv_sp=PV_SLIT1_RIGHT_SP, pv_rbv=PV_SLIT1_RIGHT_RBV,   setpoint=pos_right_open-rangex+L+dx)

        img1 = np.array(epics.caget(PV_DVF1_IMG))
        img1 = img1.reshape(NX1,NY1)

        img2 = np.array(epics.caget(PV_DVF2_IMG))
        img2 = img2.reshape(NX2,NY2)

        metadata = {
            'slit_top'    : epics.caget(PV_SLIT1_TOP_RBV),
            'slit_bottom' : epics.caget(PV_SLIT1_BOTTOM_RBV),
            'slit_left'   : epics.caget(PV_SLIT1_LEFT_RBV),
            'slit_right'  : epics.caget(PV_SLIT1_RIGHT_RBV)
        }

        file.save_dataset(dsetname=f'dvf1-{i}-{j}',dsetmetadata=metadata,dsetdata=img1)
        file.save_dataset(dsetname=f'dvf2-{i}-{j}',dsetmetadata=metadata,dsetdata=img2)

open_all_slits()

dt = time.time()-t0
print(f'ellapsed time [s]: {dt}')

Dif: 1.1389062500000016 	 New pos: 15.36 	 Curr: 16.49890625
Dif: 0.0 	 New pos: 15.36 	 Curr: 15.36
Dif: 7.105427357601002e-15 	 New pos: 46.300000000000004 	 Curr: 46.3
scan step x 1/7 and step y 1/23
scan step x 2/7 and step y 1/23
Dif: 0.0 	 New pos: 43.410000000000004 	 Curr: 43.410000000000004
Dif: 0.0 	 New pos: 46.45 	 Curr: 46.45
scan step x 3/7 and step y 1/23
Dif: 0.0 	 New pos: 43.260000000000005 	 Curr: 43.260000000000005
Dif: 0.0 	 New pos: 46.6 	 Curr: 46.6
scan step x 4/7 and step y 1/23
Dif: 0.0 	 New pos: 43.11 	 Curr: 43.11
Dif: 7.105427357601002e-15 	 New pos: 46.75000000000001 	 Curr: 46.75
scan step x 5/7 and step y 1/23
Dif: 0.0 	 New pos: 42.96 	 Curr: 42.96
Dif: 7.105427357601002e-15 	 New pos: 46.900000000000006 	 Curr: 46.9
scan step x 6/7 and step y 1/23
Dif: 0.0 	 New pos: 42.81 	 Curr: 42.81
Dif: 7.105427357601002e-15 	 New pos: 47.050000000000004 	 Curr: 47.05
scan step x 7/7 and step y 1/23
Dif: 0.0 	 New pos: 42.660000000000004 	 Curr: 42.66000000000000

In [29]:
epics.caput(PV_SLIT1_TOP_SP,18.6)
epics.caput(PV_SLIT1_BOTTOM_SP,35.5)
epics.caput(PV_SLIT1_LEFT_SP,43.5)
epics.caput(PV_SLIT1_RIGHT_SP,47)

1